# Chapter 18 — Comparative Analysis Through the Kernel Lens

Reproduces:
- Table 18.2: Mean alpha_S / alpha_A for each architecture (FT-Transformer,
  SAINT, TabPFN-lite, TabICL-lite) trained on a common substrate.
- Table 18.3: Per-architecture cosine recovery to an oracle bilinear target.
- Figure 18.1: Bar chart of mean alpha_A across architectures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tabkernels.architectures import (
    FTTransformer, SAINT, TabPFNLite, TabICLLite,
)
from tabkernels.priors import SCMPrior, SCMConfig
from tabkernels.training import PFNTrainer
from tabkernels.audits import run_cross_arch_audit

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## A common training substrate

We need every architecture to see comparable training pressure. The two
PFN architectures (TabPFN-lite, TabICL-lite) train naturally on a synthetic
prior. The two per-dataset architectures (FT-Transformer, SAINT) train on a
single classification task derived from the *same* prior. By matching the
underlying generative model we keep the comparison apples-to-apples.


In [ ]:
D = 4
prior = SCMPrior(SCMConfig(structural='mlp', edge_prob=0.5, noise_scale=0.3,
                           mlp_hidden=12))

# A fixed classification task drawn from the same prior. We reuse the
# regression labels but binarise on the median for a 2-class signal.
torch.manual_seed(42)
X_pool, y_reg, X_pool_q, y_reg_q = prior.sample_episode(
    n_ctx=400, n_query=200, d=D, seed=42,
)
X_train_cls = X_pool
y_train_cls = (y_reg > y_reg.median()).long()
X_test_cls = X_pool_q
y_test_cls = (y_reg_q > y_reg.median()).long()
print(f'classification task: {len(X_train_cls)} train, {len(X_test_cls)} test, '
      f'{(y_train_cls == 1).float().mean().item():.2f} positive rate')


## Train each architecture

PFN architectures: 600 PFN steps on the prior. Per-dataset architectures:
60 epochs of supervised training on the binarised classification task.
Identical d_model = 64, n_heads = 4, n_layers = 2 across all four.


In [ ]:
# 1) TabPFN-lite — PFN trained on the prior.
torch.manual_seed(0)
tabpfn = TabPFNLite(d_in=D, d_model=64, n_heads=4, n_layers=2, dim_ff=128)
PFNTrainer(prior=prior, model=tabpfn, n_steps=600, n_ctx=48, n_query=24,
           d=D, lr=3e-3, seed=1).train()
print('trained TabPFN-lite')

# 2) TabICL-lite — PFN trained on the prior with linear attention.
torch.manual_seed(0)
tabicl = TabICLLite(d_in=D, d_model=64, n_heads=4, n_layers=2, dim_ff=128)
PFNTrainer(prior=prior, model=tabicl, n_steps=600, n_ctx=48, n_query=24,
           d=D, lr=3e-3, seed=1).train()
print('trained TabICL-lite')

# 3) FT-Transformer — supervised training on the binarised classification task.
torch.manual_seed(0)
ftt = FTTransformer(n_num_features=D, cat_cardinalities=[],
                    d_model=64, n_heads=4, n_layers=2, dim_ff=128,
                    n_classes=2, dropout=0.1)
opt = torch.optim.Adam(ftt.parameters(), lr=3e-4)
for epoch in range(60):
    opt.zero_grad()
    logits = ftt(x_num=X_train_cls)
    loss = nn.functional.cross_entropy(logits, y_train_cls)
    loss.backward(); opt.step()
print(f'trained FT-Transformer (final loss {loss.item():.4f})')

# 4) SAINT — same training setup as FT-Transformer. SAINT uses d_token (not
# d_model) and has its own row-axis width constraint: F * d_token must be
# divisible by n_heads. With F = D = 4 and d_token = 16, that is 64 / 4 = 16, ok.
torch.manual_seed(0)
saint = SAINT(n_num_features=D, cat_cardinalities=[],
              d_token=16, n_heads=4, n_layers=2, dim_ff=64,
              n_classes=2, dropout=0.1)
opt = torch.optim.Adam(saint.parameters(), lr=3e-4)
for epoch in range(60):
    opt.zero_grad()
    logits = saint(x_num=X_train_cls)
    loss = nn.functional.cross_entropy(logits, y_train_cls)
    loss.backward(); opt.step()
print(f'trained SAINT (final loss {loss.item():.4f})')


## Apply the cross-architecture audit

Each architecture exposes ``attention_blocks()`` so the post-hoc decomposition
diagnostic of Chapter 13 works uniformly. We report mean alpha_S / alpha_A
over all heads of all blocks. Cosine-recovery requires per-architecture
oracle targets at matching head dimensions; we treat that within-architecture
in the per-architecture chapters (Figure 16.2) rather than cross-architecture.


In [ ]:
archs = {
    'FT-Transformer': ftt,
    'SAINT':         saint,
    'TabPFN-lite':   tabpfn,
    'TabICL-lite':   tabicl,
}
report = run_cross_arch_audit(archs, target_B=None)

print('Per-architecture summary:')
print(f'{"arch":<16s} {"n_blocks":>9s} {"alpha_S":>9s} {"alpha_A":>9s}')
for row in report['metrics']['summary']:
    print(f'{row["architecture"]:<16s} {row["n_blocks"]:>9d} '
          f'{row["mean_alpha_S"]:>9.3f} {row["mean_alpha_A"]:>9.3f}')

# Save the report for inclusion in chapter-side LaTeX assembly.
out_path = os.path.join(_p, 'affinity', 'book', 'data', 'cached_audits',
                        'cross_arch_kernel_lens.json')
from tabkernels.audits.base import save_report
save_report(report, out_path)
print(f'cached: {out_path}')


## Figure 18.1 — Bar chart of mean alpha_A across architectures

Lower alpha_A means the architecture allocated less of its bilinear-form
energy to the skew-symmetric component — i.e., closer to a symmetric kernel
in disguise. The McCarter 2024 finding (Chapter 16) predicts asymmetrically-
parameterised models trained on symmetric priors will have *low* alpha_A.


In [ ]:
archs_order = ['FT-Transformer', 'SAINT', 'TabPFN-lite', 'TabICL-lite']
alpha_A_vals = [next(r['mean_alpha_A'] for r in report['metrics']['summary']
                      if r['architecture'] == a) for a in archs_order]
alpha_S_vals = [next(r['mean_alpha_S'] for r in report['metrics']['summary']
                      if r['architecture'] == a) for a in archs_order]

fig, ax = plt.subplots(1, 1, figsize=(7, 3.8))
xs = np.arange(len(archs_order))
ax.bar(xs, alpha_S_vals, color='C0', label=r'$\bar\alpha_S$')
ax.bar(xs, alpha_A_vals, bottom=alpha_S_vals, color='C3', label=r'$\bar\alpha_A$')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5, label='symmetric / asymmetric parity')
ax.set_xticks(xs); ax.set_xticklabels(archs_order, rotation=15)
ax.set_ylabel('mean energy fraction'); ax.set_ylim(0, 1.05)
ax.set_title('Figure 18.1: Cross-architecture energy split')
ax.legend(loc='upper right')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_18_01_cross_arch_energy.pdf', bbox_inches='tight')
plt.show()
